# Attack detection on the Motion Light device (TON_IoT)
**Models:** Decision Tree • Random Forest • XGBoost • LightGBM • CatBoost

## Notebook steps
1. **EDA:** understand the data with plots before any model
2. **Cleaning + 10-second windows:** clean the data and group it into 10-second windows, with context from previous windows
3. **Safe split:** hash-based blocks, shared by every device, no data leakage
4. **Training:** train 5 models and compare them
5. **Deep dive:** confusion matrix, ROC, feature importance, threshold (and recall target)
6. **Block cross-validation:** check that the result is stable and not luck
7. **Multi-class:** identify the attack type
8. **Ablation + random split demo:** check that the result is honest
9. **Calibration, anomaly score and export:** unified detection events for the aggregation step

## 3 rules against leakage and overfitting
- The `type` column is not a feature because it is the label itself (same for `date` and `time`)
- The split uses contiguous time blocks, not random rows (neighbouring rows are very similar)
- Thresholds and model choices are made on validation only, and the test set is used once at the end


In [ ]:
!pip install -q catboost

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (precision_score, recall_score, f1_score, confusion_matrix,
                             roc_curve, precision_recall_curve, roc_auc_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", rc={"axes.grid.axis": "y"})
plt.rcParams["figure.dpi"] = 100
SEED = 42
SENSORS = ["motion_status"]
DATA_PATH = os.environ.get("DATA_PATH", "/content/IoT_Motion_Light.csv")

if not os.path.exists(DATA_PATH):
    from google.colab import files
    DATA_PATH = "/content/" + list(files.upload().keys())[0]

# 1) EDA

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Shape:", raw.shape)

In [ ]:
raw.head()

In [ ]:
print(raw.dtypes, "\n")

In [ ]:
print("Missing values:\n", raw.isna().sum(), "\n")

In [ ]:
print("Exact duplicates:", raw.duplicated().sum())

In [ ]:
# the text columns contain extra spaces (e.g. " on", " off"), so we strip them
for c in ["date", "time", "light_status", "type"]:
    raw[c] = raw[c].str.strip()

In [ ]:
print("Duplicates after cleaning the text columns:", raw.duplicated().sum())

In [ ]:
raw[SENSORS].describe().round(2)

In [ ]:
raw["light_status"].value_counts()

In [ ]:
raw["label"].value_counts()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
raw["label"].map({0: "Normal", 1: "Attack"}).value_counts().plot.bar(ax=ax[0], color=["#4c72b0", "#c44e52"], rot=0)
ax[0].set_title("Normal vs Attack")
raw["type"].value_counts().plot.bar(ax=ax[1], color="#55a868")
ax[1].set_yscale("log"); ax[1].set_title("Records per type (log scale)")
plt.tight_layout(); plt.show()

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<h2>Class Distribution Analysis</h2>
<h3>Normal vs Attack (left chart)</h3>
<ul>
  <li>The dataset is <b>imbalanced</b>: Normal records (~388,328) are much more than Attack records (~63,934).</li>
  <li>Normal traffic is about <b>6x</b> larger than Attack traffic.</li>
  <li>A model could get high accuracy just by predicting "Normal" every time, so accuracy alone is not a good metric.</li>
  <li>About <b>51%</b> of the rows (229,911) are exact duplicates, so cleaning matters before any grouping.</li>
</ul>
<h3>Records per type (right chart, log scale)</h3>
<ul>
  <li>The y-axis uses a <b>log scale</b> because the differences between classes are very large.</li>
  <li><b>normal</b> is the biggest class by far (~388,328 records).</li>
  <li>The attack types are also imbalanced:
    <ul>
      <li><b>backdoor</b> (~28,209) and <b>password</b> (~17,521) are the biggest attack types.</li>
      <li><b>ddos</b> (~8,121) and <b>injection</b> (~5,595) are medium-sized.</li>
      <li><b>ransomware</b> (~2,264), <b>scanning</b> (~1,775) and <b>xss</b> (~449) are the smallest classes.</li>
    </ul>
  </li>
  <li>The difference between the largest attack type and the smallest is about <b>63x</b>.</li>
</ul>
<h3>Conclusion</h3>
<ul>
  <li>The data has <b>two levels of imbalance</b>: Normal vs Attack, and between the attack types themselves.</li>
  <li>Rare classes like <b>xss</b>, <b>scanning</b> and <b>ransomware</b> will be hard for the model to learn.</li>
</ul>
</div>

In [ ]:
day = pd.to_datetime(raw["date"], format="%d-%b-%y")
att = raw[raw["label"] == 1]
pd.crosstab(day[att.index].dt.strftime("%d-%b"), att["type"]).plot.bar(stacked=True, figsize=(12, 4), colormap="tab10")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.title("Attack records per day"); plt.xlabel("Date"); plt.ylabel("Records"); plt.tight_layout(); plt.show()

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<h2>Attack Records per Day Analysis</h2>

<h3>Daily volume</h3>
<ul>
  <li>The file has records from 12 different days (31-Mar to 29-Apr), but all the attacks happen over only <b>5 days</b> (25-Apr to 29-Apr).</li>
  <li>The highest day is <b>28-Apr</b> with about <b>26,168</b> attack records.</li>
  <li>The lowest day is <b>29-Apr</b> with about <b>4,305</b> records.</li>
</ul>

<h3>Attack types by day</h3>
<ul>
  <li><b>25-Apr:</b> <b>ddos</b> (~7,192), <b>injection</b> (~5,595) &mdash; 12,787 attack records in total.</li>
  <li><b>26-Apr:</b> <b>password</b> (~11,740), <b>scanning</b> (~1,775), <b>ddos</b> (~929) &mdash; 14,444 attack records in total.</li>
  <li><b>27-Apr:</b> <b>password</b> (~5,781), <b>xss</b> (~449) &mdash; 6,230 attack records in total.</li>
  <li><b>28-Apr:</b> <b>backdoor</b> (~23,904), <b>ransomware</b> (~2,264) &mdash; 26,168 attack records in total.</li>
  <li><b>29-Apr:</b> <b>backdoor</b> (~4,305) &mdash; 4,305 attack records in total.</li>
</ul>

<h3>Key observations</h3>
<ul>
  <li>Attacks are <b>not spread evenly</b>: each attack type is concentrated in one or two specific days, like planned attack campaigns.</li>
  <li><b>injection</b>, <b>ransomware</b>, <b>scanning</b>, <b>xss</b> appear on only one day each.</li>
</ul>

<h3>Conclusion and impact on modeling</h3>
<ul>
  <li>A <b>random split</b> would mix neighbouring rows of the same campaign between train and test, and a plain <b>time-based split</b> could leave some attack types completely out of one set.</li>
  <li>The date should <b>not</b> be used as a feature, because the model may learn "which day" instead of the real attack behavior (data leakage).</li>
  <li>Rare types (<b>xss</b>, <b>scanning</b>, <b>ransomware</b>) exist in only one or two days each, so we use <b>class weights</b> and a <b>block split</b> that keeps whole time blocks together.</li>
</ul>
</div>

In [ ]:
raw["is_on"] = (raw["light_status"] == "on").astype(int)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
(raw.groupby("label")["motion_status"].mean()
    .rename(index={0: "Normal", 1: "Attack"})
    .plot.bar(ax=ax[0], color=["#4c72b0", "#c44e52"], rot=0))
ax[0].set_ylim(0, 1); ax[0].set_title("Share of motion_status = 1 by label")
order = raw["type"].value_counts().index
(raw.groupby("type")["motion_status"].mean().reindex(order)
    .plot.bar(ax=ax[1], color="#55a868", rot=45))
ax[1].set_ylim(0, 1); ax[1].set_title("Share of motion_status = 1 by type")
plt.tight_layout(); plt.show()

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<h2>Feature Distribution by Label and by Attack Type</h2>

<h3>How to read the plots</h3>
<ul>
  <li>The device has only one real signal: <code>motion_status</code> (0 = no motion, 1 = motion). <code>light_status</code> (<code>on</code> / <code>off</code>) is the same signal written as text.</li>
  <li>The left plot shows the share of rows with <code>motion_status = 1</code> for Normal and for Attack. The right plot shows the same share for every attack type.</li>
  <li>If the bars have the same height, the feature is <b>weak</b> at separating the classes.</li>
</ul>

<h3>By label</h3>
<ul>
  <li>The share of motion is <b>37.6%</b> for Normal and <b>37.4%</b> for Attack, so there is no difference.</li>
</ul>

<h3>By attack type</h3>
<ul>
  <li>Every type is between <b>36.3%</b> and <b>37.7%</b>, including normal (37.6%).</li>
</ul>

<h3>Conclusion</h3>
<ul>
  <li>The value of a single row <b>cannot separate</b> Attack from Normal, and no attack type has its own motion pattern.</li>
  <li>The difference must come from <b>other signals</b>, like how many rows arrive in a time window, and how the state changes over time.</li>
</ul>
</div>

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(raw[SENSORS + ["is_on", "label"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation"); plt.show()

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<h2>Correlation Analysis</h2>

<h3>How to read the heatmap</h3>
<ul>
  <li>Values range from <b>-1 to +1</b>: +1 is a perfect positive relation, -1 is a perfect negative relation, and <b>0 means no linear relation</b>.</li>
  <li>The diagonal is always 1.00 because each column is compared with itself.</li>
  <li><code>is_on</code> is <code>light_status</code> turned into 0/1 (1 = on).</li>
</ul>

<h3>Features vs each other</h3>
<ul>
  <li><b>motion_status</b> and <b>is_on</b> have a correlation of <b>1.00</b>: the light is on exactly when there is motion.</li>
  <li>So <code>light_status</code> is a <b>duplicate</b> of <code>motion_status</code>, and we do not use it in the window features.</li>
</ul>

<h3>Features vs label</h3>
<ul>
  <li>Both have a correlation of <b>-0.00</b> with the label (almost exactly zero).</li>
  <li>There is <b>no linear relationship</b> between the raw values and the label.</li>
</ul>

<h3>Conclusion</h3>
<ul>
  <li>The raw row feature is a <b>very weak predictor</b>, which matches the previous plots.</li>
  <li>The useful signal for this device is expected to be in the <b>window level</b> (number of records, how often the state flips, history), so we build window features next.</li>
</ul>
</div>

In [ ]:
rts = pd.to_datetime(raw["date"] + " " + raw["time"], format="%d-%b-%y %H:%M:%S")
first = raw.index[raw["label"] == 1][0]
seg = raw.iloc[max(first - 200, 0) : first + 1500].reset_index(drop=True)
seg["gap_s"] = rts.iloc[max(first - 200, 0) : first + 1500].diff().dt.total_seconds().clip(-5, 30).values   # seconds since the previous row
fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for a, c in zip(ax, SENSORS + ["gap_s"]):
    a.plot(seg[c], lw=0.7, color="gray")
    a.scatter(seg.index[seg.label == 1], seg.loc[seg.label == 1, c], s=4, color="red")
    a.set_ylabel(c)
ax[-1].set_xlabel("Row order"); plt.tight_layout(); plt.show()

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<h2>Sensor Values by Row Order (Time Series)</h2>

<h3>How to read the plot</h3>
<ul>
  <li>The top panel shows <b>motion_status</b> and the bottom panel shows <b>gap_s</b> (seconds since the previous row) across the row order of the data. The plot starts 200 rows before the first attack.</li>
  <li>The <b>gray line</b> is the value, and the <b>red dots</b> mark the <b>Attack</b> rows (label 1).</li>
</ul>

<h3>What we see</h3>
<ul>
  <li><b>Before the red dots</b> the data is normal. <b>After them</b> the attack rows are mixed with normal rows.</li>
  <li><b>motion_status</b> flips between 0 and 1 all the time, both before and after the attack starts. Its average is <b>0.38</b> on the normal rows and <b>0.38</b> on the attack rows of this part.</li>
  <li>The red dots appear <b>on both values</b> (0 and 1), so a single value does not tell us that a row is an attack.</li>
</ul>

<h3>Conclusion and impact on modeling</h3>
<ul>
  <li>The attack is not visible in the value itself; it can only change <b>how the records arrive</b> (the number of rows per time window).</li>
  <li>Do <b>not</b> use row index or date as a feature, because it can cause <b>data leakage</b>.</li>
  <li>Use <b>10-second window features</b> and <b>lag / rolling features</b> from the previous windows.</li>
</ul>
</div>

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<h2>EDA Summary: What We Learned</h2>

<ul>
  <li><b>Imbalanced data:</b> Normal records are about <b>6x</b> more than Attack records, and some attack types (<b>xss</b>, <b>scanning</b>, <b>ransomware</b>) are very rare.</li>
  <li><b>Attacks depend on the day:</b> all attacks are between 25-Apr and 29-Apr, and each attack type appears in only one or two specific days.</li>
  <li><b>No signal in the raw value:</b> the share of motion is 37.6% for Normal and 37.4% for Attack, and its correlation with the label is <b>-0.00</b>.</li>
  <li><b>No difference between attack types:</b> every type has almost the same share of motion (36.3% to 37.7%).</li>
  <li><b>Redundant feature:</b> <code>light_status</code> is exactly the same as <code>motion_status</code> (correlation 1.00), so it adds nothing.</li>
  <li><b>Dirty data:</b> the text columns have extra spaces, and about 51% of the rows are duplicates, so cleaning is important before any grouping.</li>
  <li><b>The real signal is in the timing:</b> the number of records per window is different between Normal and Attack (see step 2), so the window and history features are the key for this device.</li>
</ul>
</div>

# 2) Cleaning + 10-Second Windows

In [ ]:
df = raw.drop_duplicates().reset_index(drop=True)
print(f"Rows: {len(raw)} -> {len(df)} after removing duplicates")
#Remove exact duplicate rows and print the number of rows before and after

In [ ]:
ts = pd.to_datetime(df["date"] + " " + df["time"], format="%d-%b-%y %H:%M:%S")
#Combine date and time into one timestamp ts

In [ ]:
df["chunk"] = (ts.diff().dt.total_seconds() < 0).cumsum()
sec = (ts - pd.Timestamp("2019-01-01")).dt.total_seconds()
df["sec"] = sec
df["win"] = (sec // 10).astype(int)
#The file is made of several time parts stuck together. When the time goes backward, a new part chunk starts.
#Chunks keep us from mixing rows that are not really consecutive.
#Create 10-second windows
#Convert the time to seconds and divide by 10, so all rows in the same 10 seconds get the same win number

In [ ]:
# a "flip" = the motion state changed compared with the previous row of the same window
df["flip"] = (df.groupby(["chunk", "win"], sort=False)["motion_status"].diff().fillna(0) != 0).astype(int)

g = df.groupby(["chunk", "win"], sort=False)
W = g[SENSORS].agg(["mean", "std", "min", "max"])
W.columns = [f"{a}_{b}" for a, b in W.columns]
W["n_changes"] = g["flip"].sum()
W["n_records"] = g.size()
W["span_s"] = g["sec"].max() - g["sec"].min()
#For the sensor (motion_status) we calculate mean, std, min, max in each window.
#n_changes is how many times the motion state flips inside the window.
#n_records is the number of rows in the window. It can help detect attacks like ddos that change the traffic.
#span_s is the time between the first and the last row inside the window.
#light_status is not used as a feature: it is exactly the same signal as motion_status (correlation 1.00), so it adds nothing.

In [ ]:
# Count how many rows of each type (normal, ddos, ...) are inside each window
counts = pd.crosstab([df["chunk"], df["win"]], df["type"]).reindex(W.index)
# The window label = the most common type inside the window (majority vote)
W["type"] = counts.idxmax(axis=1).values
# Binary label: 1 if the window is an attack, 0 if it is normal
W["is_attack"] = (W["type"] != "normal").astype(int)
# Turn chunk and win from index back into normal columns, and fill missing values with 0
W = W.reset_index().fillna(0)

In [ ]:
# Block number (same block size as the split in step 3). All the history below is taken inside the
# SAME (chunk, block) only, so no feature uses information from a neighbouring block (which can be in another set).
W["block"] = W["win"] // 100
gb = W.groupby(["chunk", "block"])

# Number of records in the previous window (same chunk and block only)
W["prev_n_records"] = gb["n_records"].shift(1).fillna(0)
# Time gap (in windows) between this window and the previous one
W["gap_prev"] = gb["win"].diff().fillna(0)

In [ ]:
# Loop over the main columns we want to add history for
for c in ["motion_status_mean", "motion_status_std", "n_changes", "n_records", "span_s"]:

    # Lag features: the value of this column 1, 2, and 3 windows ago
    for k in (1, 2, 3):
        W[f"{c}_lag{k}"] = gb[c].shift(k)

    # Rolling mean: average of the previous 3 windows (shift(1) excludes the current window)
    W[f"{c}_roll3"] = gb[c].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

    # Change feature: current value minus the recent average (shows sudden changes)
    W[f"{c}_chg"] = W[c] - W[f"{c}_roll3"]

# Longer history: average of the previous 10 windows and of everything before it in the block.
# Attacks last many windows in a row, so this is a strong signal.
for c in ["motion_status_mean", "n_changes", "n_records", "span_s"]:
    W[f"{c}_roll10"] = gb[c].transform(lambda s: s.shift(1).rolling(10, min_periods=1).mean())
    W[f"{c}_expand"] = gb[c].transform(lambda s: s.shift(1).expanding().mean())

# Fill the missing values (first windows of each block have no history) with 0
W = W.fillna(0)

# Use all columns as features except the identifiers (chunk, win, block) and the labels (type, is_attack)
FEATURES = [c for c in W.columns if c not in ("chunk", "win", "block", "type", "is_attack")]

# Print: number of windows, percentage of attack windows, and number of features
print(f"Windows: {len(W)} | Attack windows: {W.is_attack.mean():.1%} | Features: {len(FEATURES)}")

In [ ]:
W.head()

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(data=W[W.n_records <= 30], x="n_records", hue=W["is_attack"].map({0: "Normal", 1: "Attack"}),
             stat="density", common_norm=False, discrete=True, palette=["#4c72b0", "#c44e52"], element="step")
plt.title("Records per 10-second window"); plt.show()

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<h2>Result: Records per 10-Second Window</h2>
<ul>
  <li><b>Normal windows</b> are mostly at <b>6</b> (21.4%), <b>5</b> (20.8%), <b>4</b> (15.3%) records, and the average is <b>4.8</b> records per window.</li>
  <li><b>Attack windows</b> are mostly at <b>3</b> (23.5%), <b>4</b> (21.9%), <b>2</b> (19.6%) records, and the average is <b>3.5</b> records per window.</li>
  <li>Windows with <b>5 to 7 records</b> are mostly Normal, and windows with <b>1 to 4 records</b> are mostly Attack, so rows are often <b>missing</b> during attacks.</li>
  <li>The two classes <b>overlap</b> at 4 and 5 records, so <code>n_records</code> is useful but <b>not enough alone</b>.</li>
  <li>The state also flips less often during attacks: <b>1.35</b> flips per window for Attack and <b>2.09</b> for Normal (<code>n_changes</code>).</li>
  <li>This supports using <b>n_records</b>, <b>n_changes</b> and their <b>lag / rolling</b> versions as features.</li>
</ul>
</div>

In [ ]:
run_id = (W["is_attack"] != W["is_attack"].shift()).cumsum()
run_len = W[W.is_attack == 1].groupby(run_id).size()
plt.figure(figsize=(9, 3.5))
plt.hist(run_len.clip(upper=60), bins=30, color="#c44e52")
plt.xlabel("Consecutive attack windows in a row (clipped at 60)"); plt.ylabel("Count of attack runs")
plt.title("How long do attacks last?"); plt.show()
print(f"Median run = {run_len.median():.0f} windows | longest = {run_len.max()} windows")

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<h2>How Long Do Attacks Last?</h2>

<h3>How to read the plot</h3>
<ul>
  <li>X-axis: the number of <b>consecutive attack windows</b> in a row (each window = 10 seconds). Values above 60 are <b>clipped</b> to 60.</li>
  <li>Y-axis: the <b>count of attack runs</b> (how many separate attack periods have this length).</li>
</ul>

<h3>What we see</h3>
<ul>
  <li><b>Very short attacks:</b> 41 of the 184 runs (22%) last only 1-2 windows.</li>
  <li><b>Very long attacks:</b> 36 runs (20%) last <b>60 windows or more</b>, which means 10 minutes or more.</li>
  <li>The median run is <b>11 windows</b> (1.8 minutes) and the longest run is <b>2,327 windows</b> (about 6.5 hours).</li>
</ul>

<h3>Conclusion</h3>
<ul>
  <li>Attacks usually <b>continue for many windows</b>, which supports using <b>lag and rolling features</b> from previous windows (we also use a longer 10-window history).</li>
  <li>Very short attacks (1-2 windows) have little history, so they will be <b>harder to detect</b>.</li>
  <li>Neighboring windows inside one attack are very similar, so the train/test split should be by <b>blocks</b>, not random, to avoid leakage.</li>
</ul>
</div>

# 3) Safe Split

In [ ]:
# ---------- Hash-based block split (same rule on every device) ----------
import hashlib

BLOCK = 100                              # windows per block
SALT = "iot-anomaly"                     # shared salt, identical on every device
TRAIN_PCT, VAL_PCT = 70, 15              # hash buckets: [0,70) train, [70,85) val, [85,100) test

n = len(W)                               # total number of windows
all_idx = np.arange(n)                   # index of every window
types = W["type"].values                 # attack type of every window
n_types = W["type"].nunique()            # number of types

def assign_split(win, bw=BLOCK, salt=SALT):
    block = np.asarray(win) // bw        # absolute block number counted from the epoch, same on every device
    bucket = {b: int(hashlib.md5(f"{salt}:{b}".encode()).hexdigest(), 16) % 100 for b in np.unique(block)}
    r = np.array([bucket[b] for b in block])
    return np.where(r < TRAIN_PCT, "train", np.where(r < TRAIN_PCT + VAL_PCT, "val", "test"))

block_id = W["win"].values // BLOCK
split_name = assign_split(W["win"].values)
train, val, test = (np.where(split_name == s)[0] for s in ("train", "val", "test"))

# Remove the first 3 and last 3 windows of every block (safety gap),
# because lag / rolling features look at the previous 3 windows
pos = W["win"].values % BLOCK
edge = (pos < 3) | (pos >= BLOCK - 3)
train, val, test = [i[~edge[i]] for i in (train, val, test)]

# Safety check: no block is shared between two sets
assert not set(block_id[train]) & set(block_id[val])
assert not set(block_id[train]) & set(block_id[test])
assert not set(block_id[val])   & set(block_id[test])

# Features (X) and labels (y)
X, y = W[FEATURES], W["is_attack"].values
y_tr, y_va, y_te = y[train], y[val], y[test]

# Summary 1: number of windows and attack percentage in each set
display(pd.DataFrame(
    {"windows":  [len(train), len(val), len(test)],
     "attack %": [y_tr.mean() * 100, y_va.mean() * 100, y_te.mean() * 100]},
    index=["train", "val", "test"]
).round(1))

# Summary 2: number of windows of each type in each set
pd.DataFrame({
    "train": W["type"].iloc[train].value_counts(),
    "val":   W["type"].iloc[val].value_counts(),
    "test":  W["type"].iloc[test].value_counts(),
}).fillna(0).astype(int)

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<ul>
  <li>There is <b>no leakage</b>, because every block stays in only one set (checked by the asserts), and 3,069 edge windows (first 3 and last 3 of every block) were removed.</li>
  <li>The attack percentage is different between the sets (train 33.7%, val 26.4%, test 37.4%), because whole blocks (and so whole attack campaigns) go to one set.</li>
  <li>The model can learn <b>xss</b> (154 train windows) but it <b>cannot be evaluated</b> on it, because there is no xss window in the test set.</li>
  <li><b>injection</b> has only 1 window in the test set, so its multi-class result is not meaningful.</li>
  <li><b>scanning</b> has only 15 windows in the validation set, so model selection for the multi-class task is based on few examples of it.</li>
  <li>The history features (lag, rolling, expanding) are computed inside the same block only, so they never use a neighbouring block.</li>
  <li>For the binary target (<code>is_attack</code>) the effect is small, since validation and test contain many attack windows of other types.</li>
</ul>
</div>

# 4) Model

In [ ]:
spw = (y_tr == 0).sum() / (y_tr == 1).sum()

def make_models(spw):
    return {
        "DecisionTree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, class_weight="balanced", random_state=SEED),
        "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=10,
                                               class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
        "XGBoost": XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, min_child_weight=5,
                                 subsample=0.8, colsample_bytree=0.8, reg_lambda=5, scale_pos_weight=spw,
                                 early_stopping_rounds=50, eval_metric="aucpr", n_jobs=-1, random_state=SEED),
        "LightGBM": LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5,
                                   class_weight="balanced", n_jobs=-1, random_state=SEED, verbose=-1),
        "CatBoost": CatBoostClassifier(iterations=600, learning_rate=0.08, depth=6, l2_leaf_reg=5,
                                       auto_class_weights="Balanced", early_stopping_rounds=50,
                                       random_seed=SEED, verbose=0),
    }

def fit(name, model, Xtr, ytr, Xva, yva, sample_weight=None):

    if name == "XGBoost":
        model.fit(Xtr, ytr, sample_weight=sample_weight, eval_set=[(Xva, yva)], verbose=False)
    elif name == "LightGBM":
        model.fit(Xtr, ytr, sample_weight=sample_weight, eval_set=[(Xva, yva)], callbacks=[early_stopping(50, verbose=False)])
    elif name == "CatBoost":
        model.fit(Xtr, ytr, sample_weight=sample_weight, eval_set=(Xva, yva))
    else:
        model.fit(Xtr, ytr, sample_weight=sample_weight)
    return model

def best_threshold(y_true, proba):
    ths = np.linspace(0.05, 0.95, 91)
    return ths[np.argmax([f1_score(y_true, proba >= t) for t in ths])]

In [ ]:
models, probs, rows = {}, {}, []
for name, m in make_models(spw).items():
    m = fit(name, m, X.iloc[train], y_tr, X.iloc[val], y_va)
    p_tr, p_va, p_te = (m.predict_proba(X.iloc[i])[:, 1] for i in (train, val, test))
    th = best_threshold(y_va, p_va)                       # threshold chosen on validation only
    models[name], probs[name] = m, dict(train=p_tr, val=p_va, test=p_te, th=th)

    # same metrics for train and test
    for split, y_true, p in (("train", y_tr, p_tr), ("test", y_te, p_te)):
        rows.append(dict(model=name, split=split, threshold=round(th, 2),
                         recall=recall_score(y_true, p >= th),
                         precision=precision_score(y_true, p >= th),
                         f1=f1_score(y_true, p >= th),
                         auc=roc_auc_score(y_true, p)))

results = pd.DataFrame(rows).round(3)

# side-by-side table: train vs test for each model
cmp = results.pivot(index="model", columns="split", values=["recall", "precision", "f1", "auc"])
cmp[("f1", "gap")] = cmp[("f1", "train")] - cmp[("f1", "test")]   # overfitting gap
cmp.round(3)

# 5) Best model analysis

In [ ]:
# Choose the best model by validation F1 (the test set is not used for selection)
best = max(probs, key=lambda n: f1_score(y_va, probs[n]["val"] >= probs[n]["th"]))
print("Best model by validation F1:", best)

pt, th = probs[best]["test"], probs[best]["th"]   # test probabilities and threshold of the best model

In [ ]:
# Probability calibration: fitted on validation only.
# Class weights distort raw probabilities, so we map them to calibrated attack probabilities.
from sklearn.isotonic import IsotonicRegression

calibrator = IsotonicRegression(out_of_bounds="clip").fit(probs[best]["val"], y_va)

p_cal_va = calibrator.predict(probs[best]["val"])
p_cal_te = calibrator.predict(probs[best]["test"])
th_cal = best_threshold(y_va, p_cal_va)            # threshold chosen on validation only

print(f"Calibrated threshold: {th_cal:.2f}")
print(f"Test F1 (calibrated): {f1_score(y_te, p_cal_te >= th_cal):.3f}")

In [ ]:
# Confusion matrix of the best model
cm = confusion_matrix(y_te, pt >= th)

plt.figure(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title(f"{best} — Confusion Matrix")
plt.tight_layout(); plt.show()

In [ ]:
# ROC curve for all models
plt.figure(figsize=(6, 5))
for nm, d in probs.items():
    fpr, tpr, _ = roc_curve(y_te, d["test"])
    plt.plot(fpr, tpr, label=nm)
plt.plot([0, 1], [0, 1], "k--", lw=0.7)       # random guess line
plt.title("ROC"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Precision-Recall curve for all models
plt.figure(figsize=(6, 5))
for nm, d in probs.items():
    p, r, _ = precision_recall_curve(y_te, d["test"])
    plt.plot(r, p, label=nm)
plt.title("Precision-Recall"); plt.xlabel("Recall"); plt.ylabel("Precision"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
imp = pd.Series(models[best].feature_importances_, index=FEATURES).sort_values(ascending=False)
imp.head(15)[::-1].plot.barh(figsize=(8, 6), color="#55a868")
plt.title(f"{best} feature importance (top 15)")
plt.xlabel("Importance")
plt.tight_layout(); plt.show()

In [ ]:
ths = np.linspace(0.02, 0.98, 97)
rec = [recall_score(y_te, pt >= t) for t in ths]
pre = [precision_score(y_te, pt >= t, zero_division=1) for t in ths]
f1s = [f1_score(y_te, pt >= t) for t in ths]
plt.figure(figsize=(9, 4))
plt.plot(ths, rec, label="Recall"); plt.plot(ths, pre, label="Precision"); plt.plot(ths, f1s, label="F1", ls="--")
plt.axvline(th, color="gray", ls=":", label=f"chosen on validation ({th:.2f})")
plt.xlabel("Threshold"); plt.legend(); plt.title(f"{best}: threshold trade-off"); plt.show()

pv = probs[best]["val"]; out = []
for target in (0.90, 0.95):
    ok = [t for t in ths if recall_score(y_va, pv >= t) >= target]
    t = max(ok) if ok else ths[0]
    out.append(dict(target_recall=target, threshold=round(t, 2), test_recall=recall_score(y_te, pt >= t),
                    test_precision=precision_score(y_te, pt >= t), test_f1=f1_score(y_te, pt >= t)))
pd.DataFrame(out).round(3)

# 6) Cross-Validation

In [ ]:
cv_rows = []
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (tr_i, te_i) in enumerate(outer.split(all_idx, W["type"].values, groups=block_id)):
    tr_i, te_i = tr_i[~edge[tr_i]], te_i[~edge[te_i]]
    inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    a, b = next(inner.split(tr_i, W["type"].values[tr_i], groups=block_id[tr_i]))
    fit_i, va_i = tr_i[a], tr_i[b]
    spw_k = (y[fit_i] == 0).sum() / (y[fit_i] == 1).sum()
    for name, m in make_models(spw_k).items():
        m = fit(name, m, X.iloc[fit_i], y[fit_i], X.iloc[va_i], y[va_i])
        th_k = best_threshold(y[va_i], m.predict_proba(X.iloc[va_i])[:, 1])
        pred = m.predict_proba(X.iloc[te_i])[:, 1] >= th_k
        cv_rows.append(dict(fold=fold, model=name, recall=recall_score(y[te_i], pred),
                            precision=precision_score(y[te_i], pred), f1=f1_score(y[te_i], pred)))
    print("fold", fold, "done")
cv_df = pd.DataFrame(cv_rows)
cv_summary = cv_df.groupby("model")[["recall", "precision", "f1"]].agg(["mean", "std"]).round(3)
cv_summary

# 7) Multi-Class:

In [ ]:
le = LabelEncoder().fit(W["type"]); classes = list(le.classes_)
yc = le.transform(W["type"]); yc_tr, yc_va, yc_te = yc[train], yc[val], yc[test]
sw = compute_sample_weight("balanced", yc_tr)
print("Test support:", dict(pd.Series(W["type"].values[test]).value_counts()))

mc = {
    "DecisionTree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, class_weight="balanced", random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=10, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=600, learning_rate=0.07, max_depth=5, min_child_weight=3, subsample=0.8, colsample_bytree=0.8,
                             reg_lambda=5, early_stopping_rounds=40, n_jobs=-1, random_state=SEED),
    "LightGBM": LGBMClassifier(n_estimators=600, learning_rate=0.07, num_leaves=31, min_child_samples=15, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.8, reg_lambda=5, class_weight="balanced", n_jobs=-1, random_state=SEED, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, l2_leaf_reg=5, loss_function="MultiClass",
                                   auto_class_weights="Balanced", early_stopping_rounds=40, random_seed=SEED, verbose=0),
}
present = sorted(set(yc_te)); mc_pred, mc_rows = {}, []
for name, m in mc.items():

    m = fit(name, m, X.iloc[train], yc_tr, X.iloc[val], yc_va, sample_weight=sw if name == "XGBoost" else None)
    pred = np.asarray(m.predict(X.iloc[test])).ravel().astype(int); mc_pred[name] = pred
    mc_rows.append(dict(model=name, macro_f1=f1_score(yc_te, pred, labels=present, average="macro", zero_division=0),
                        weighted_f1=f1_score(yc_te, pred, average="weighted", zero_division=0)))
mc_results = pd.DataFrame(mc_rows).set_index("model").round(3)
mc_results

In [ ]:
# choose the multi-class model by validation macro-F1 (the test set is not used for selection)
val_f1 = {n: f1_score(yc_va, np.asarray(m.predict(X.iloc[val])).ravel().astype(int),
                      labels=sorted(set(yc_va)), average="macro") for n, m in mc.items()}
best_mc = max(val_f1, key=val_f1.get); print("Best multi-class model:", best_mc)
pred = mc_pred[best_mc]

cm = confusion_matrix(yc_te, pred, labels=range(len(classes)))[present]
cm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", ax=ax[0], xticklabels=classes, yticklabels=[classes[i] for i in present])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual"); ax[0].set_title(f"{best_mc} — Confusion Matrix (row-normalised)")

per_class = pd.DataFrame({m: [(p[yc_te == k] == k).mean() for k in present] for m, p in mc_pred.items()},
                         index=[classes[i] for i in present])
per_class.plot.bar(ax=ax[1], rot=45, colormap="Set2"); ax[1].set_title("Recall per attack type"); ax[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

# 8) Baselines (Alsaedi et al.)

<style>
.wbox, .wbox * { color:#111111 !important; }
.wbox { background-color:#ffffff !important; border:6px solid #ffffff; border-radius:12px; padding:18px 26px; font-family:Arial, sans-serif; line-height:1.7; box-shadow:0 0 0 1px #cccccc; }
.wbox h2, .wbox h3 { margin-top:10px; }
</style>
<div class="wbox">
<h2>Baseline models from the TON_IoT paper</h2>
<ul>
  <li>These are the baseline models used in the TON_IoT paper (Alsaedi et al.): <b>LR, LDA, kNN (k=5), CART, RF (10 trees), NB and SVM (RBF kernel)</b>.</li>
  <li>They are trained on the <b>same windows, features, split and protocol</b> as the 5 main models (threshold chosen on validation only, test used once), for both the <b>binary</b> and the <b>multi-class</b> task.</li>
  <li>Models that need scaled inputs (LR, kNN, SVM) use a scaler fitted on the <b>training windows only</b>, so there is no leakage.</li>
  <li>The 8th baseline of the paper, <b>LSTM</b>, is <b>not trained</b> on purpose, until we decide about it.</li>
  <li>This section is only for <b>comparison</b>: it does not change the model chosen in the export step.</li>
</ul>
</div>

In [ ]:
# Baseline models from the TON_IoT paper (Alsaedi et al.): LR, LDA, kNN, CART, RF, NB, SVM (RBF).
# The 8th baseline of the paper (LSTM, deep learning) is intentionally NOT trained here.
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

def make_baselines():
    scaled = lambda m: make_pipeline(StandardScaler(), m)      # the scaler is fitted on the training windows only
    return {
        "LR":   scaled(LogisticRegression(max_iter=1000, class_weight="balanced")),
        "LDA":  LinearDiscriminantAnalysis(),
        "kNN":  scaled(KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),     # k = 5
        "CART": DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
        "RF":   RandomForestClassifier(n_estimators=10, class_weight="balanced", n_jobs=-1, random_state=SEED),   # 10 trees
        "NB":   GaussianNB(),
        "SVM":  scaled(SVC(kernel="rbf", class_weight="balanced")),
    }

def attack_score(m, Xs):
    # probability if the model has one, otherwise the SVM decision value passed through a sigmoid (same ranking)
    if hasattr(m, "predict_proba"):
        return m.predict_proba(Xs)[:, 1]
    return 1 / (1 + np.exp(-m.decision_function(Xs)))

In [ ]:
# Binary (Normal vs Attack): same protocol as the 5 main models (threshold chosen on validation only)
base_rows, base_probs = [], {}
for name, m in make_baselines().items():
    m.fit(X.iloc[train], y_tr)
    p_tr, p_va, p_te = (attack_score(m, X.iloc[i]) for i in (train, val, test))
    th_b = best_threshold(y_va, p_va)
    base_probs[name] = dict(test=p_te, th=th_b)
    base_rows.append(dict(model=name, threshold=round(th_b, 2),
                          train_f1=f1_score(y_tr, p_tr >= th_b), val_f1=f1_score(y_va, p_va >= th_b),
                          test_recall=recall_score(y_te, p_te >= th_b), test_precision=precision_score(y_te, p_te >= th_b),
                          test_f1=f1_score(y_te, p_te >= th_b), test_auc=roc_auc_score(y_te, p_te)))
base_results = pd.DataFrame(base_rows).set_index("model").round(3)
base_results

In [ ]:
# Multi-class (attack type): same classes and the same macro-F1 rule as the 5 main models
base_mc_pred, base_mc_rows = {}, []
for name, m in make_baselines().items():
    m.fit(X.iloc[train], yc_tr)
    pred = np.asarray(m.predict(X.iloc[test])).ravel().astype(int); base_mc_pred[name] = pred
    base_mc_rows.append(dict(model=name,
                             macro_f1=f1_score(yc_te, pred, labels=present, average="macro", zero_division=0),
                             weighted_f1=f1_score(yc_te, pred, average="weighted", zero_division=0)))
base_mc_results = pd.DataFrame(base_mc_rows).set_index("model").round(3)
base_mc_results

In [ ]:
# Baselines vs the 5 main models (Test F1)
main_bin = {n: f1_score(y_te, d["test"] >= d["th"]) for n, d in probs.items()}
cmp_all = pd.DataFrame({
    "binary F1": pd.concat([base_results["test_f1"], pd.Series(main_bin)]),
    "multi-class macro-F1": pd.concat([base_mc_results["macro_f1"], mc_results["macro_f1"]]),
}).round(3)
cmp_all["group"] = ["baseline (paper)"] * len(base_results) + ["main model"] * len(probs)
cmp_all.plot.bar(y=["binary F1", "multi-class macro-F1"], figsize=(11, 4), rot=30, colormap="Set2")
plt.ylim(0, 1); plt.title("Baselines (paper) vs main models: Test F1"); plt.tight_layout(); plt.show()
cmp_all

# 9) (Ablation + Random Split)

In [ ]:

TIMING = [c for c in FEATURES if "n_records" in c or "span_s" in c or c == "gap_prev"]
def quick_f1(cols):
    m = fit("XGBoost", make_models(spw)["XGBoost"], X[cols].iloc[train], y_tr, X[cols].iloc[val], y_va)
    th_ = best_threshold(y_va, m.predict_proba(X[cols].iloc[val])[:, 1])
    pr = m.predict_proba(X[cols].iloc[test])[:, 1] >= th_
    return dict(recall=recall_score(y_te, pr), precision=precision_score(y_te, pr), f1=f1_score(y_te, pr))

abl = pd.DataFrame({"all features": quick_f1(FEATURES),
                    "sensors only (no counts)": quick_f1([c for c in FEATURES if c not in TIMING]),
                    "counts/timing only": quick_f1(TIMING)}).T.round(3)
abl.plot.bar(rot=15, figsize=(9, 4), colormap="Set2"); plt.ylim(0, 1); plt.title("Ablation (XGBoost, Test)"); plt.show()
abl

In [ ]:
a, b = train_test_split(all_idx, test_size=0.2, stratify=y, random_state=SEED)
a, v = train_test_split(a, test_size=0.2, stratify=y[a], random_state=SEED)

rows = []
for name, m in make_models(spw).items():
    m = fit(name, m, X.iloc[a], y[a], X.iloc[v], y[v])
    th_ = best_threshold(y[v], m.predict_proba(X.iloc[v])[:, 1])
    rows.append(dict(model=name,
                     random_split=f1_score(y[b], m.predict_proba(X.iloc[b])[:, 1] >= th_),

                     block_split=f1_score(y_te, probs[name]["test"] >= probs[name]["th"])))
cmp = pd.DataFrame(rows).set_index("model").round(3)
cmp.plot.bar(rot=0, figsize=(9, 4), color=["#c44e52", "#4c72b0"])
plt.ylim(0, 1); plt.ylabel("Test F1"); plt.title("Random split (leaky) vs Block split (honest)"); plt.show()
print(cmp)

In [ ]:
os.makedirs("results", exist_ok=True)
results.to_csv("results/binary_results.csv"); mc_results.to_csv("results/multiclass_results.csv")
abl.to_csv("results/ablation.csv"); base_results.to_csv("results/baseline_binary.csv"); base_mc_results.to_csv("results/baseline_multiclass.csv"); cv_summary.to_csv("results/cv_summary.csv"); per_class.to_csv("results/recall_per_attack_type.csv"); cmp.to_csv("results/random_vs_block.csv")
print("Saved in ./results")

# 10) Anomaly score (Z-score, Mahalanobis, Isolation Forest, Autoencoder)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
from sklearn.ensemble import IsolationForest

norm_tr = train[y[train] == 0]                      # Normal windows of the train set only

def to_percentile(raw):
    # convert a raw score to 0..1 (percentile compared with Normal train windows)
    ref = np.sort(raw[norm_tr])
    return np.searchsorted(ref, raw) / len(ref)

# 1) Z-score: average distance (in standard deviations) from the Normal mean
sc = StandardScaler().fit(X.iloc[norm_tr])
raw_z = np.abs(sc.transform(X)).mean(axis=1)

# 2) Mahalanobis: distance from the Normal distribution (uses correlation between features)
cov = EmpiricalCovariance().fit(X.iloc[norm_tr])
raw_m = cov.mahalanobis(X)

# 3) Isolation Forest
iso = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X.iloc[norm_tr])
raw_i = -iso.score_samples(X)

scores = {"Z-score": to_percentile(raw_z),
          "Mahalanobis": to_percentile(raw_m),
          "IsolationForest": to_percentile(raw_i)}
print("done:", list(scores))

In [ ]:
# Autoencoder anomaly score: learns to rebuild Normal windows, the reconstruction error is the score
from sklearn.neural_network import MLPRegressor

sc_ae = StandardScaler().fit(X.iloc[norm_tr])       # fit on Normal train windows only
Xn = sc_ae.transform(X.iloc[norm_tr])

ae = MLPRegressor(hidden_layer_sizes=(32, 16, 8, 16, 32), activation="relu", solver="adam",
                  max_iter=200, early_stopping=True, n_iter_no_change=10, random_state=SEED)
ae.fit(Xn, Xn)                                      # input = target

X_all = sc_ae.transform(X)
raw_ae = ((X_all - ae.predict(X_all)) ** 2).mean(axis=1)   # mean squared reconstruction error

scores["Autoencoder"] = to_percentile(raw_ae)
print("done:", list(scores))

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
                             recall_score, precision_score, f1_score, confusion_matrix)

ths = np.linspace(0.01, 0.99, 99)
# threshold of each model: best F1 on validation only
th_an = {nm: ths[np.argmax([f1_score(y[val], s[val] >= t) for t in ths])] for nm, s in scores.items()}

rows = []
for nm, s in scores.items():
    for split_nm, ii in (("val", val), ("test", test)):
        pred = s[ii] >= th_an[nm]
        rows.append(dict(model=nm, split=split_nm, threshold=round(th_an[nm], 2),
                         auc=roc_auc_score(y[ii], s[ii]),
                         avg_precision=average_precision_score(y[ii], s[ii]),
                         baseline_ap=y[ii].mean(),                    # AP of a random score = attack rate
                         recall=recall_score(y[ii], pred),
                         precision=precision_score(y[ii], pred, zero_division=0),
                         f1=f1_score(y[ii], pred)))
anom_results = pd.DataFrame(rows).set_index(["model", "split"]).round(3)
anom_results

In [ ]:
# Pick the anomaly method with the best validation F1 and keep what is needed to reuse it
best_an = anom_results.xs("val", level="split")["f1"].idxmax()
anomaly = scores[best_an]
print("Anomaly method:", best_an)

# 11) Export

In [ ]:
# Final export: unified detection events + config
import json, os
os.makedirs("export", exist_ok=True); os.makedirs("results", exist_ok=True)

SOURCE = "motion_light"
mc_best = mc[best_mc]
th_final = float(th_cal)

# calibrated attack probability for every window (the same scale as th_final)
pa = calibrator.predict(models[best].predict_proba(X)[:, 1])
pred_type = le.inverse_transform(np.asarray(mc_best.predict(X)).ravel().astype(int))
is_attack_pred = pa >= th_final
# attack type only for windows predicted as Attack ("unknown" if the multi-class model says normal)
attack_type = np.where(is_attack_pred, np.where(pred_type != "normal", pred_type, "unknown"), "none")

# purged edge windows are marked here, then removed from both exported files
split = np.full(len(W), "purged", dtype=object)
for idx, key in ((train, "train"), (val, "val"), (test, "test")):
    split[idx] = key

events = pd.DataFrame({
    "source": SOURCE,
    "window_start": pd.Timestamp("2019-01-01") + pd.to_timedelta(W["win"] * 10, unit="s"),
    "prediction": np.where(is_attack_pred, "Attack", "Normal"),
    "attack_prob": pa,
    "attack_type": attack_type,
    "anomaly_score": anomaly,
    "n_records": W["n_records"].values,
    "true_label": W["is_attack"].values,
    "split": split,
})
# events file: purged edge windows are NOT exported (same rule as the predictions file)
keep = split != "purged"
events = events.loc[keep].copy()
events.insert(0, "event_id", np.arange(len(events), dtype=int))
events.to_csv(f"export/{SOURCE}_events.csv", index=False)
anom_results.to_csv("results/anomaly_results.csv")

# predictions file (purged windows removed). p_attack is the calibrated probability, so
# "p_attack >= threshold" in the config gives exactly the predictions above.
export_df = pd.DataFrame({"source": SOURCE, "chunk": W["chunk"], "win": W["win"], "split": split,
                          "p_attack": pa, "y_true": W["is_attack"], "true_type": W["type"],
                          "pred_type": pred_type, "n_records": W["n_records"]})
export_df[export_df["split"] != "purged"].to_csv(f"export/{SOURCE}_predictions.csv", index=False)

cfg = {"multiclass_model": best_mc, "binary_model": best, "source": SOURCE, "anomaly_method": best_an,
       "calibrated": True, "threshold": th_final, "epoch": "2019-01-01", "window_seconds": 10,
       "classes": classes, "features": FEATURES,
       "test_f1": float(f1_score(y_te, p_cal_te >= th_final)),
       "val_f1": float(f1_score(y_va, p_cal_va >= th_final)),
       "split_rule": {"method": "md5 hash of block number", "salt": SALT, "train_pct": TRAIN_PCT, "val_pct": VAL_PCT},
       "block_windows": BLOCK}
with open(f"export/{SOURCE}_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

# ---------- read-back checks (what we exported is exactly what the aggregation step will read) ----------
PRED_COLS = ["source", "chunk", "win", "split", "p_attack", "y_true", "true_type", "pred_type", "n_records"]
chk = pd.read_csv(f"export/{SOURCE}_predictions.csv")
assert list(chk.columns) == PRED_COLS, "predictions columns / order are wrong"
assert set(chk["split"]) <= {"train", "val", "test"}, "purged rows are still in predictions"
assert chk["p_attack"].between(0, 1).all() and chk["p_attack"].notna().all()
assert not chk.duplicated(["chunk", "win"]).any(), "duplicate (chunk, win) rows"
blk = chk["win"] // BLOCK
assert blk.groupby(blk).ngroups == chk.groupby(blk)["split"].nunique().shape[0] and (chk.groupby(blk)["split"].nunique() == 1).all(), "a block is in two sets"
assert not ((chk["win"] % BLOCK < 3) | (chk["win"] % BLOCK >= BLOCK - 3)).any(), "edge windows were not purged"

ev = pd.read_csv(f"export/{SOURCE}_events.csv")
assert "purged" not in set(ev["split"]), "purged rows are still in events"
assert len(ev) == len(chk), "events and predictions must have the same rows"
assert ((chk["p_attack"].values >= th_final) == (ev["prediction"].values == "Attack")).all(), "p_attack >= threshold must give the events prediction"
assert ev["anomaly_score"].between(0, 1).all()

with open(f"export/{SOURCE}_config.json") as f:
    cfg_back = json.load(f)
for k in ["multiclass_model", "binary_model", "source", "anomaly_method", "calibrated", "threshold", "epoch", "window_seconds",
          "classes", "features", "test_f1", "val_f1", "split_rule", "block_windows"]:
    assert k in cfg_back, f"missing config key: {k}"
print("export checks passed")

print(cfg["binary_model"], cfg["multiclass_model"], best_an, "| val F1:", round(cfg["val_f1"], 3), "| test F1:", round(cfg["test_f1"], 3))
print(events.groupby("split").size().to_dict())
events.head()